# T2.5 - DBRepo Data Load

Owner: Person C

Verify the T2.1 schema in 3NF, load raw data into DBRepo, verify T2.4 views.

## 1. Schema Verification

The T2.1 schema has three tables: station, time_dimension, weather_measurement_v2.

**1NF:** Atomic columns, single primary keys (station_num, time_id, measurement_id). No repeating groups.

**2NF:** Station metadata (nuts_code, coordinates) moved to station table - depends only on station_num, not on time. Time metadata (year, month) moved to time_dimension - depends only on time_id, not on station. No partial dependencies on the composite key.

**3NF:** No transitive dependencies. In station, all columns depend directly on station_num. In time_dimension, all columns depend directly on time_id. In weather_measurement, measurements depend on the observation identified by measurement_id.

The raw CSV has 29 flat columns. Station codes (NUTS, DISTRICT_CODE, SUB_DISTRICT_CODE) repeat across every row for the same station. Year/month metadata repeats across every row for the same month. Under 3NF, these are separated into the dimension tables, with weather_measurement holding only observations and foreign keys. This eliminates redundancy while preserving all information.

**Conclusion:** The schema correctly models the data in 3NF.

In [75]:
import os
from getpass import getpass
import pandas as pd

from dbrepo.RestClient import RestClient
from dbrepo.api.dto import QueryDefinition

In [77]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

DOTENV_PATH = find_dotenv()
load_dotenv(DOTENV_PATH, override=True)
REPO_ROOT = Path(DOTENV_PATH).resolve().parent if DOTENV_PATH else Path.cwd()

ENDPOINT = os.getenv("DBREPO_ENDPOINT", "https://test.dbrepo.tuwien.ac.at")
DATABASE_ID = os.getenv("DBREPO_DATABASE_ID")
USERNAME = os.getenv("DBREPO_USERNAME") or input("DBRepo username: ")
PASSWORD = os.getenv("DBREPO_PASSWORD") or getpass("DBRepo password: ")

if not USERNAME:
    raise RuntimeError("DBREPO_USERNAME is not configured.")
if not PASSWORD:
    raise RuntimeError("DBREPO_PASSWORD is not configured.")
if not DATABASE_ID:
    raise RuntimeError("DBREPO_DATABASE_ID is not configured.")

TABLE_IDS = {
    "weather_measurement_v2": os.getenv("DBREPO_TABLE_WEATHER_MEASUREMENT_ID"),
    "time_dimension": os.getenv("DBREPO_TABLE_TIME_DIMENSION_ID"),
    "station": os.getenv("DBREPO_TABLE_STATION_ID"),
}

missing_table_ids = [name for name, table_id in TABLE_IDS.items() if not table_id]
if missing_table_ids:
    raise RuntimeError(
        "Missing DBRepo table ID environment variable(s) for: "
        + ", ".join(missing_table_ids)
    )

client = RestClient(
    endpoint=ENDPOINT,
    username=USERNAME,
    password=PASSWORD,
)

print("Connected to DBRepo.")

Connected to DBRepo.


## 2. Read Raw CSV

File has a description header on row 1, column names on row 2. Uses ; as delimiter and , as decimal separator.

In [80]:
raw_df = pd.read_csv(
    REPO_ROOT / "data" / "raw" / "weather_raw_vienna_hohewarte_v1.csv",
    sep=";", decimal=",",
)
print(f"Shape: {raw_df.shape}, Columns: {list(raw_df.columns)}")
print(f"Years: {raw_df['REF_YEAR'].min()}-{raw_df['REF_YEAR'].max()}")
print(f"Stations: {sorted(raw_df['STAT_NUM'].unique())}")
raw_df.head(2)

Shape: (1847, 29), Columns: ['NUTS', 'DISTRICT_CODE', 'SUB_DISTRICT_CODE', 'REF_YEAR', 'REF_DATE', 'T', 'T_MAX', 'T_MIN', 'MEAN_T_MAX', 'MEAN_T_MIN', 'NUM_FROST', 'NUM_ICE', 'NUM_SUMMER', 'NUM_HEAT', 'P', 'P_MAX', 'P_MIN', 'SUN_H', 'NUM_CLEAR', 'NUM_CLOUD', 'REL_HUM', 'REL_HUM_MAX', 'REL_HUM_MIN', 'WIND_VEL', 'NUM_WIND_VEL60', 'WIND_VEL_MAX', 'PRECP_SUM', 'NUM_PRECP_01', 'STAT_NUM']
Years: 1872-2026
Stations: [np.int64(5901), np.int64(5904)]


,NUTS,DISTRICT_CODE,SUB_DISTRICT_CODE,REF_YEAR,REF_DATE,T,T_MAX,T_MIN,MEAN_T_MAX,MEAN_T_MIN,...,NUM_CLOUD,REL_HUM,REL_HUM_MAX,REL_HUM_MIN,WIND_VEL,NUM_WIND_VEL60,WIND_VEL_MAX,PRECP_SUM,NUM_PRECP_01,STAT_NUM
0,AT13,91900,91905,1872,187205,17.0,30.6,7.9,23.1,11.8,...,9,64,NaN,NaN,7.2,NaN,NaN,50,8,5901
1,AT13,91900,91905,1872,187206,17.2,28.3,10.0,22.6,12.9,...,7,66,NaN,NaN,10.8,NaN,NaN,73,11,5901


## 3. Transform into 3NF Tables

In [81]:
# 3.1 Station table
station_df = (
    raw_df[["STAT_NUM", "NUTS", "DISTRICT_CODE", "SUB_DISTRICT_CODE"]]
    .drop_duplicates()
    .rename(columns={
        "STAT_NUM": "station_num", "NUTS": "nuts_code",
        "DISTRICT_CODE": "district_code", "SUB_DISTRICT_CODE": "sub_district_code"
    })
)
station_df["station_name"] = "Wien - Hohe Warte"
station_df["latitude_deg"] = 48.248611
station_df["longitude_deg"] = 16.356944
station_df["altitude_m"] = 202.0
station_df = station_df.drop_duplicates(subset=["station_num"]).set_index("station_num")
print(f"Station rows: {len(station_df)}")
station_df

Station rows: 2


,nuts_code,district_code,sub_district_code,station_name,latitude_deg,longitude_deg,altitude_m
station_num,,,,,,,
5901,AT13,91900,91905,Wien - Hohe Warte,48.248611,16.356944,202.0
5904,AT13,91900,91905,Wien - Hohe Warte,48.248611,16.356944,202.0


In [51]:
# 3.2 Time dimension table
raw_df["ref_month"] = pd.to_datetime(raw_df["REF_DATE"].astype(str), format="%Y%m").dt.month
raw_df["ref_year"] = raw_df["REF_YEAR"].astype(int)
raw_df["time_id"] = raw_df["ref_year"] * 100 + raw_df["ref_month"]

time_df = (
    raw_df[["time_id", "ref_year", "ref_month"]]
    .drop_duplicates().sort_values("time_id").set_index("time_id")
)
print(f"Time dimension rows: {len(time_df)}, range: {time_df.index.min()}-{time_df.index.max()}")
time_df.head(3)

Time dimension rows: 1847, range: 187205-202603


,ref_year,ref_month
time_id,,
187205,1872,5
187206,1872,6
187207,1872,7


In [52]:
# 3.3 Weather measurement table
weather_df = raw_df.copy()
weather_df["station_num"] = weather_df["STAT_NUM"].astype(int)
weather_df["time_id"] = weather_df["ref_year"] * 100 + weather_df["ref_month"]

weather_df = weather_df.rename(columns={
    "T": "t_mean_c", "T_MAX": "t_max_c", "T_MIN": "t_min_c",
    "MEAN_T_MAX": "mean_t_max_c", "MEAN_T_MIN": "mean_t_min_c",
    "P": "p_mean_hpa", "P_MAX": "p_max_hpa", "P_MIN": "p_min_hpa",
    "PRECP_SUM": "precp_sum_mm", "NUM_PRECP_01": "num_precp_01",
    "REL_HUM": "rel_hum_pct", "REL_HUM_MAX": "rel_hum_max_pct", "REL_HUM_MIN": "rel_hum_min_pct",
    "WIND_VEL": "wind_vel_ms", "WIND_VEL_MAX": "wind_vel_max_ms", "NUM_WIND_VEL60": "num_wind_vel60",
    "SUN_H": "sun_h", "NUM_CLEAR": "num_clear", "NUM_CLOUD": "num_cloud",
    "NUM_FROST": "num_frost", "NUM_ICE": "num_ice", "NUM_SUMMER": "num_summer", "NUM_HEAT": "num_heat",
})

meas_cols = ["station_num", "time_id",
    "t_mean_c", "t_max_c", "t_min_c", "mean_t_max_c", "mean_t_min_c",
    "p_mean_hpa", "p_max_hpa", "p_min_hpa",
    "precp_sum_mm", "num_precp_01",
    "rel_hum_pct", "rel_hum_max_pct", "rel_hum_min_pct",
    "wind_vel_ms", "wind_vel_max_ms", "num_wind_vel60",
    "sun_h", "num_clear", "num_cloud",
    "num_frost", "num_ice", "num_summer", "num_heat"]

weather_df = weather_df[meas_cols].sort_values(["station_num", "time_id"]).reset_index(drop=True)
weather_df.insert(0, "measurement_id", range(1, len(weather_df) + 1))
weather_df = weather_df.set_index("measurement_id")
print(f"Weather rows: {len(weather_df)}")

Weather rows: 1847


## 4. Data Types and Missing Values

The dataset contains structural missing values. For example, `REL_HUM_MAX`, `REL_HUM_MIN`, `WIND_VEL_MAX`, and `NUM_WIND_VEL60` are missing before 1951, while `SUN_H` is missing before 1921 because the corresponding measurements were not available in the historical data.

These missing values are preserved as `NULL` in DBRepo. They are not replaced by zero, because `NULL` means “not measured / not available”, while `0` would mean an actual measured value of zero.

The corrected DBRepo table `weather_measurement_v2` allows missing values for these historical columns. Some nullable numeric columns are represented as text by DBRepo schema inference, so they are converted back to numeric in later processing/T2.6.

In [53]:
int_cols = [
    "station_num",
    "time_id",
    "num_precp_01",
    "num_clear",
    "num_cloud",
    "num_frost",
    "num_ice",
    "num_summer",
    "num_heat",
]

nullable_int_cols = [
    "num_wind_vel60",
]

dec_cols = [
    "t_mean_c",
    "t_max_c",
    "t_min_c",
    "mean_t_max_c",
    "mean_t_min_c",
    "p_mean_hpa",
    "p_max_hpa",
    "p_min_hpa",
    "precp_sum_mm",
    "rel_hum_pct",
    "rel_hum_max_pct",
    "rel_hum_min_pct",
    "wind_vel_ms",
    "wind_vel_max_ms",
    "sun_h",
]

for col in int_cols:
    weather_df[col] = pd.to_numeric(weather_df[col], errors="coerce").astype("int64")

for col in nullable_int_cols:
    weather_df[col] = pd.to_numeric(weather_df[col], errors="coerce").astype("Int64")

for col in dec_cols:
    weather_df[col] = pd.to_numeric(weather_df[col], errors="coerce")

nulls = weather_df.isnull().sum()
print("Null counts preserved:")
print(nulls[nulls > 0] if nulls.any() else "No nulls found")

Null counts preserved:
p_mean_hpa           2
p_max_hpa            2
p_min_hpa            2
rel_hum_max_pct    944
rel_hum_min_pct    944
wind_vel_max_ms    944
num_wind_vel60     944
sun_h               99
dtype: int64


## 5. Referential Consistency

In [54]:
assert weather_df.index.is_unique and time_df.index.is_unique and station_df.index.is_unique
assert set(weather_df["time_id"]).issubset(set(time_df.index))
assert set(weather_df["station_num"]).issubset(set(station_df.index))

print(f"Referential checks passed.")
print(f"  station: {len(station_df)}, time: {len(time_df)}, weather: {len(weather_df)}")

Referential checks passed.
  station: 2, time: 1847, weather: 1847


## 6. Upload to DBRepo

In [55]:
for name, tid in TABLE_IDS.items():
    tbl = client.get_table(DATABASE_ID, tid)
    print(f"{name}: {[c.name for c in tbl.columns]}")

weather_measurement_v2: ['measurement_id', 'station_num', 'time_id', 't_mean_c', 't_max_c', 't_min_c', 'mean_t_max_c', 'mean_t_min_c', 'p_mean_hpa', 'p_max_hpa', 'p_min_hpa', 'precp_sum_mm', 'num_precp_01', 'rel_hum_pct', 'rel_hum_max_pct', 'rel_hum_min_pct', 'wind_vel_ms', 'wind_vel_max_ms', 'num_wind_vel60', 'sun_h', 'num_clear', 'num_cloud', 'num_frost', 'num_ice', 'num_summer', 'num_heat']
time_dimension: ['time_id', 'ref_year', 'ref_month']
station: ['station_num', 'nuts_code', 'district_code', 'sub_district_code', 'station_name', 'latitude_deg', 'longitude_deg', 'altitude_m']


In [56]:
# Upload station
try:
    sdf = station_df.reset_index()[["station_num", "nuts_code", "district_code",
        "sub_district_code", "station_name", "latitude_deg", "longitude_deg", "altitude_m"]]
    client.import_table_data(DATABASE_ID, TABLE_IDS["station"], sdf)
    print("station uploaded")
except Exception as e:
    print(f"station upload (may already exist): {e}")


station uploaded


In [64]:
# Upload time_dimension
time_df = (
    weather_df[["time_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

time_df["ref_year"] = time_df["time_id"] // 100
time_df["ref_month"] = time_df["time_id"] % 100

time_df = time_df[["time_id", "ref_year", "ref_month"]]

try:
    tdf = time_df.reset_index()[["time_id", "ref_year", "ref_month"]]
    client.import_table_data(DATABASE_ID, TABLE_IDS["time_dimension"], tdf)
    print("time_dimension uploaded")
except Exception as e:
    print(f"time_dimension upload (may already exist): {e}")


time_dimension uploaded


In [63]:
# Upload weather_measurement
before = len(weather_df)

pressure_required_cols = [
    "p_mean_hpa",
    "p_max_hpa",
    "p_min_hpa",
]

weather_df = weather_df.dropna(subset=pressure_required_cols)

after = len(weather_df)

print(f"Dropped {before - after} rows with missing pressure values.")
print(f"Remaining rows: {after}")

try:
    wdf = weather_df.reset_index()
    wdf = wdf[["measurement_id", "station_num", "time_id",
        "t_mean_c", "t_max_c", "t_min_c", "mean_t_max_c", "mean_t_min_c",
        "p_mean_hpa", "p_max_hpa", "p_min_hpa",
        "precp_sum_mm", "num_precp_01",
        "rel_hum_pct", "rel_hum_max_pct", "rel_hum_min_pct",
        "wind_vel_ms", "wind_vel_max_ms", "num_wind_vel60",
        "sun_h", "num_clear", "num_cloud",
        "num_frost", "num_ice", "num_summer", "num_heat"]]
    client.import_table_data(DATABASE_ID, TABLE_IDS["weather_measurement_v2"], wdf)
    print("weather_measurement uploaded")
except Exception as e:
    print(f"weather_measurement upload error: {e}")


Dropped 2 rows with missing pressure values.
Remaining rows: 1845
weather_measurement uploaded


## 7. Verify Row Counts

In [70]:
print("Row counts after upload:")
for name, tid in TABLE_IDS.items():
    cnt = client.get_table_data_count(DATABASE_ID, tid)
    print(f"  {name}: {cnt}")

Row counts after upload:
  weather_measurement_v2: 1845
  time_dimension: 1845
  station: 2


## 8. Verify T2.4 View

In [73]:
views = client.get_views(DATABASE_ID)
for v in views:
    vc = client.get_view_data_count(DATABASE_ID, v.id)
    print(f"{v.name}: {vc} rows")

view = next(v for v in views if v.name == "weather_measurement_v2_features")
view_count = client.get_view_data_count(DATABASE_ID, view.id)
table_count = client.get_table_data_count(DATABASE_ID, TABLE_IDS["weather_measurement_v2"])
print(f"\nView == Table rows? {view_count == table_count} ({view_count} vs {table_count})")

weather_measurement_v2_features: 1845 rows

View == Table rows? True (1845 vs 1845)


## 9. Save Processed Files

In [74]:
processed_dir = REPO_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

station_df.reset_index().to_csv(processed_dir / "station_v1.csv", index=False)
time_df.reset_index().to_csv(processed_dir / "time_dimension_v1.csv", index=False)
weather_df.reset_index().to_csv(processed_dir / "weather_measurement_v1.csv", index=False)
print(f"Saved to {processed_dir}")

Saved to data/processed/
